In [ ]:
# the purpose of this notebook is to find the most uncertain formulas

In [2]:
# import data

from pathlib import Path
import pandas as pd

# Load the compiled data
project_root = Path.cwd().resolve().parent if 'notebooks' in str(Path.cwd()) else Path.cwd().resolve()
excel_path = project_root / 'data' / 'raw' / 'Compiled Data.xlsx'

df = pd.read_excel(excel_path)


# Show the 25 original formulas and their three measured targets from df.
formula_cols = ["TEM", "Joncryl", "NC513", "PEG-DE", "NGDE"]
measured_target_cols = [
    "Avg_Density_g_cm3",
    "24h_avg_pct absorption",
    "24h_avg_pct_retention",
]
df = df[formula_cols + measured_target_cols].copy()
df[formula_cols] = df[formula_cols].fillna(0.0)
df[measured_target_cols] = df[measured_target_cols].apply(
    pd.to_numeric, errors="raise"
)

assert len(df) == 25, (
    f"Expected the 25 original samples; found {len(df)}."
)
display(df)

,TEM,Joncryl,NC513,PEG-DE,NGDE,Avg_Density_g_cm3,24h_avg_pct absorption,24h_avg_pct_retention
0,8,0.0,0.0,0.0,0.0,0.192000,113.616943,31.770000
1,8,0.7,0.0,0.0,0.0,0.195000,103.817309,33.120000
2,8,1.0,0.0,0.0,0.0,0.159000,129.464292,36.250000
3,8,1.5,0.0,0.0,0.0,0.159933,120.749405,25.330000
4,8,2.0,0.0,0.0,0.0,0.201000,113.871581,26.580000
5,8,3.0,0.0,0.0,0.0,0.202000,111.044347,27.400000
6,5,0.0,0.0,0.0,0.0,0.298930,56.958538,36.614852
7,5,0.0,0.7,0.0,0.0,0.360035,57.115160,44.587460
8,5,0.0,1.0,0.0,0.0,0.355557,44.308895,43.766577
9,5,0.0,2.0,0.0,0.0,0.358055,43.326301,44.186536


In [31]:
# get the 10,000 candidate formulas, they are stored in candidate_set

# Formula constraints:
# - TEM: 5-15%.
# - Each chain extender (Joncryl, NC513, PEG-DE, NGDE): 0.7-3% when included; 0% when omitted.
# - Combined chain extender amount: no more than 6%.
# - Generate 10,000 valid candidate formulas.

import numpy as np
import pandas as pd

candidate_count = 10_000
chain_extender_cols = ["Joncryl", "NC513", "PEG-DE", "NGDE"]
feature_cols = ["TEM", *chain_extender_cols]
rng = np.random.default_rng(42)

candidate_chunks = []
collected = 0
while collected < candidate_count:
    batch_size = max(1_000, 2 * (candidate_count - collected))
    active_counts = rng.integers(1, len(chain_extender_cols) + 1, size=batch_size)
    active_order = rng.random((batch_size, len(chain_extender_cols))).argsort(axis=1)
    active_mask = active_order < active_counts[:, None]
    extender_amounts = rng.uniform(0.7, 3.0, size=(batch_size, len(chain_extender_cols)))
    extender_amounts = np.where(active_mask, extender_amounts, 0.0)

    valid_mask = extender_amounts.sum(axis=1) <= 6.0
    valid_batch = extender_amounts[valid_mask]
    candidate_chunks.append(valid_batch)
    collected += len(valid_batch)

extender_values = np.concatenate(candidate_chunks, axis=0)[:candidate_count]
candidate_set = pd.DataFrame(extender_values, columns=chain_extender_cols)
candidate_set.insert(0, "TEM", rng.uniform(5.0, 15.0, size=candidate_count))
candidate_set = candidate_set[feature_cols]

# Confirm the generated formulas satisfy every requested bound.
assert len(candidate_set) == candidate_count
assert candidate_set["TEM"].between(5.0, 15.0).all()
assert candidate_set[chain_extender_cols].sum(axis=1).le(6.0).all()
for col in chain_extender_cols:
    active = candidate_set[col].ne(0)
    assert candidate_set.loc[active, col].between(0.7, 3.0).all()

print(f"Generated {len(candidate_set):,} valid candidate formulas.")
print(candidate_set.head())
print("Maximum total chain extender:", candidate_set[chain_extender_cols].sum(axis=1).max())

Generated 10,000 valid candidate formulas.
         TEM   Joncryl     NC513    PEG-DE      NGDE
0   6.518998  0.000000  0.000000  0.000000  1.372423
1  14.665367  1.690923  2.920128  0.000000  0.754151
2   7.065769  2.274083  0.000000  2.099238  0.000000
3  11.644699  0.000000  1.788971  0.000000  0.832839
4  14.939813  0.000000  0.000000  2.326068  0.000000
Maximum total chain extender: 5.998874020416366


In [32]:
# get predictions and predictive uncertainty for these formulas, stored in
# candidate_predictions

import joblib
from pathlib import Path

project_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
models_dir = project_root / "models"
gpr_models = joblib.load(models_dir / "gpr_models.joblib")

feature_cols = ["TEM", "Joncryl", "NC513", "PEG-DE", "NGDE"]
X_candidates = candidate_set.loc[:, feature_cols]

batch_size = 5_000
prediction_batches = {name: {"mean": [], "std": []} for name in gpr_models}

for name, model in gpr_models.items():
    for start in range(0, len(X_candidates), batch_size):
        batch = X_candidates.iloc[start : start + batch_size]
        mean_batch, std_batch = model.predict(batch, return_std=True)
        prediction_batches[name]["mean"].append(mean_batch)
        prediction_batches[name]["std"].append(std_batch)

prediction_columns = {}
for name, batches in prediction_batches.items():
    prediction_columns[f"pred_{name}"] = np.concatenate(batches["mean"])
    prediction_columns[f"std_{name}"] = np.concatenate(batches["std"])

predictions = pd.DataFrame(prediction_columns, index=candidate_set.index)
candidate_predictions = pd.concat([candidate_set, predictions], axis=1)

required_std_cols = [f"std_{name}" for name in gpr_models]
missing_std_cols = [col for col in required_std_cols if col not in candidate_predictions.columns]
if missing_std_cols:
    raise KeyError(f"Prediction standard-deviation columns were not created: {missing_std_cols}")

display(candidate_predictions.head())


,TEM,Joncryl,NC513,PEG-DE,NGDE,pred_Avg_Density_g_cm3,std_Avg_Density_g_cm3,pred_24h_avg_pct absorption,std_24h_avg_pct absorption,pred_24h_avg_pct_retention,std_24h_avg_pct_retention
0,6.518998,0.000000,0.000000,0.000000,1.372423,0.245038,0.052154,71.028928,21.610494,33.664603,13.062179
1,14.665367,1.690923,2.920128,0.000000,0.754151,0.214560,0.064126,178.837075,118.883859,31.248809,15.401931
2,7.065769,2.274083,0.000000,2.099238,0.000000,0.214329,0.045233,166.123166,38.201897,32.682255,14.347879
3,11.644699,0.000000,1.788971,0.000000,0.832839,0.214145,0.064124,176.057859,70.546098,31.187843,15.396372
4,14.939813,0.000000,0.000000,2.326068,0.000000,0.214560,0.064126,246.952351,129.736294,31.245747,15.401931


In [33]:
# find the 10 most uncertain formulas (highest std)

import numpy as np

std_cols = [col for col in candidate_predictions.columns if col.startswith("std_")]
if not std_cols:
    raise ValueError("No standard-deviation columns (std_...) found in candidate_predictions.")

std_values = candidate_predictions[std_cols].to_numpy(dtype=float)
if not np.isfinite(std_values).all():
    raise ValueError("Standard-deviation values contain missing or non-finite values.")

std_min = std_values.min(axis=0)
std_range = std_values.max(axis=0) - std_min

# Normalize each property's uncertainty to [0, 1].
normalized_std = np.divide(
    std_values - std_min,
    std_range,
    out=np.zeros_like(std_values),
    where=std_range > 0,
)

df = candidate_predictions.copy()
df["uncertainty_score"] = normalized_std.mean(axis=1)

# The 10,000 formulas with the highest equal-weight normalized uncertainty.
df = df.nlargest(10000, "uncertainty_score").copy()

display(df)

,TEM,Joncryl,NC513,PEG-DE,NGDE,pred_Avg_Density_g_cm3,std_Avg_Density_g_cm3,pred_24h_avg_pct absorption,std_24h_avg_pct absorption,pred_24h_avg_pct_retention,std_24h_avg_pct_retention,uncertainty_score
4683,14.987895,2.004696,0.000000,2.961479,0.796613,0.214560,0.064126,201.114109,134.545178,31.248563,15.401931,1.000000
1652,14.898653,2.502876,0.000000,2.821847,0.000000,0.214560,0.064126,197.616808,133.791537,31.248515,15.401931,0.997841
8603,14.944530,2.424502,0.000000,2.540242,0.000000,0.214560,0.064126,203.228995,132.669262,31.248486,15.401931,0.994626
8944,14.911651,1.232669,1.302349,2.952429,0.000000,0.214560,0.064126,215.541696,132.653770,31.248376,15.401931,0.994582
8780,14.900000,2.954067,0.000000,2.389941,0.000000,0.214560,0.064126,196.823713,132.539794,31.248270,15.401931,0.994255
...,...,...,...,...,...,...,...,...,...,...,...,...
7177,8.004472,0.000000,0.000000,0.000000,2.050236,0.224031,0.012959,97.908696,18.976392,36.505204,2.745551,0.009164
4714,8.025928,0.000000,0.000000,0.000000,1.845852,0.219364,0.013016,101.024002,18.700556,36.015541,2.745172,0.008727
4681,8.012242,0.000000,0.000000,0.000000,1.939813,0.221499,0.012966,99.498540,18.817190,36.248187,2.733700,0.008444
3874,7.957935,0.000000,0.000000,0.000000,1.441680,0.211918,0.013023,104.653964,18.308860,35.060731,2.758302,0.007998


In [34]:
# save the models predictions (most uncertain formulations to predictions folder)

from pathlib import Path

project_root = (
    Path.cwd().resolve().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd().resolve()
)

predictions_dir = project_root / "predictions"
predictions_dir.mkdir(parents=True, exist_ok=True)

output_path = predictions_dir / "top_uncertain_formulas.xlsx"
df.to_excel(output_path, index=False)

print(f"Saved Excel file to: {output_path}")

Saved Excel file to: C:\Users\Brooke\Foam project\predictions\top_uncertain_formulas.xlsx
